# SHIPP — Validate Agent `save_item` Boundary

`Gold candidate → current Lakebase status → idempotent saved_items write → agent_activity audit`

**Read-only by default.** The only cell that writes is gated by `RUN_WRITE_TEST`, which stays `False` until every gate above it passes.

## Before you run
1. `psycopg[binary]` must be in this notebook's **Environment** panel (it already is). Do **not** add `%pip` + `restartPython()` after `%run`: the restart wipes everything the bootstrap loaded.
2. Fill in the two widgets at the top: the SHIPP Lakebase **instance** and the **database** that contains schema `shipp`.

## Gates
| Gate | Checks | Writes? |
|---|---|---|
| 0 | bootstrap + agent imports | no |
| 1 | Lakebase connection opens and sees `shipp` | no |
| 2 | unique constraint + `action_status` values match the Agent | no |
| 3 | Gold has every column the Agent reads (needs Gold built) | no |
| 4 | one deterministic Gold candidate exists in Lakebase | no |
| 5 | save → duplicate → rejected, each confirmed by SELECT | **yes, only if RUN_WRITE_TEST** |

In [0]:
%uv pip install "psycopg[binary]>=3.2,<4"

In [0]:
import psycopg

print("psycopg version:", psycopg.__version__)
print("PASS — psycopg available")

In [0]:
%run ../common/shipp_silver_lib

In [0]:
# Gate 0a — SHIPP Lakebase Autoscaling parameters.
# Read-only configuration. No database writes.

import os
import sys
import json
import psycopg

from databricks.sdk import WorkspaceClient

RUN_WRITE_TEST = False  # SAFETY SWITCH — keep False until Gates 0–4 PASS

ENDPOINT_NAME = (
    "projects/shipp-capstone-v1/"
    "branches/production/endpoints/primary"
)

PGHOST = (
    "ep-dry-flower-d17p00tu."
    "database.us-west-2.cloud.databricks.com"
)

PGDATABASE = "databricks_postgres"
LAKEBASE_SCHEMA = "shipp"

w = WorkspaceClient()
PGUSER = w.current_user.me().user_name


def connect():
    """Open a fresh authenticated connection to SHIPP Lakebase Autoscaling."""
    credential = w.postgres.generate_database_credential(
        endpoint=ENDPOINT_NAME
    )

    return psycopg.connect(
        host=PGHOST,
        port=5432,
        dbname=PGDATABASE,
        user=PGUSER,
        password=credential.token,
        sslmode="require",
        connect_timeout=10,
    )


print("RUN_WRITE_TEST =", RUN_WRITE_TEST)
print("Endpoint       =", ENDPOINT_NAME)
print("Database       =", PGDATABASE)
print("User           =", PGUSER)
print("Schema         =", LAKEBASE_SCHEMA)
print("PASS Gate 0a — Autoscaling target configured")

In [0]:
# Gate 0b — import the Agent package.
# No database writes.

agent_src = os.path.join(
    REPO_ROOT,
    "agent",
    "agent_ship",
    "src",
)

if agent_src not in sys.path:
    sys.path.insert(0, agent_src)

from shipp.agent.contracts import (
    ActionStatus,
    CANDIDATE_MATCH_COLUMNS,
)

from shipp.agent.lakebase import LakebaseRepo

print(
    "PASS Gate 0 — agent package imported from",
    agent_src,
)

In [0]:
# Gate 1a — prove this endpoint is the real SHIPP Lakebase database.
# READ ONLY.

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT
                current_user,
                current_database(),
                EXISTS (
                    SELECT 1
                    FROM information_schema.schemata
                    WHERE schema_name = 'shipp'
                ),
                to_regclass('shipp.listings'),
                to_regclass('shipp.requests'),
                to_regclass('shipp.saved_items'),
                to_regclass('shipp.agent_activity')
        """)

        row = cur.fetchone()

pg_user = row[0]
pg_database = row[1]
has_shipp_schema = row[2]
listings_table = row[3]
requests_table = row[4]
saved_items_table = row[5]
agent_activity_table = row[6]

print("User           :", pg_user)
print("Database       :", pg_database)
print("SHIPP schema   :", has_shipp_schema)
print("listings       :", listings_table)
print("requests       :", requests_table)
print("saved_items    :", saved_items_table)
print("agent_activity :", agent_activity_table)

assert pg_database == PGDATABASE, (
    f"Expected database {PGDATABASE}, got {pg_database}"
)

assert has_shipp_schema, (
    "BLOCKED: schema 'shipp' does not exist."
)

assert listings_table is not None
assert requests_table is not None
assert saved_items_table is not None
assert agent_activity_table is not None

print(
    "PASS Gate 1a — confirmed SHIPP operational database"
)

In [0]:
# Gate 1b — bind Agent repository to verified Autoscaling connection.

from shipp.agent.lakebase import LakebaseRepo

lakebase = LakebaseRepo(
    connect,
    LAKEBASE_SCHEMA,
)

with connect() as conn:
    with conn.cursor() as cur:
        cur.execute(
            f"SELECT count(*) FROM {LAKEBASE_SCHEMA}.listings"
        )
        listing_count = cur.fetchone()[0]

print("Listings in Lakebase:", listing_count)
print("PASS Gate 1 — LakebaseRepo connected to SHIPP Autoscaling")

In [0]:
# Read-only check for a non-AVAILABLE listing before Gate 5d.
with connect() as conn, conn.cursor() as cur:
    cur.execute(
        f"""
        SELECT listing_id, status
        FROM {LAKEBASE_SCHEMA}.listings
        ORDER BY listing_id
        """
    )
    rows = cur.fetchall()

print("Lakebase listings:")
for row in rows:
    print(row)

non_available = [
    row for row in rows
    if str(row[1]).upper() != "AVAILABLE"
]

print("\nNon-AVAILABLE listings:", non_available)

In [0]:
# Gate 2 — real PostgreSQL constraints must match what the Agent writes. Read-only.
def constraints(table):
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            "SELECT conname, pg_get_constraintdef(oid) FROM pg_constraint "
            "WHERE conrelid = %s::regclass ORDER BY conname",
            (f"{LAKEBASE_SCHEMA}.{table}",),
        )
        return dict(cur.fetchall())

saved_constraints = constraints("saved_items")
activity_constraints = constraints("agent_activity")

print("saved_items constraints:")
for n, d in saved_constraints.items():
    print("  ", n, "=>", d)
print("agent_activity constraints:")
for n, d in activity_constraints.items():
    print("  ", n, "=>", d)

# 3a — ON CONFLICT (user_id, request_id, listing_id) needs a UNIQUE on exactly these columns.
# Checked by definition, not by name, so a renamed migration still passes.
def is_save_unique(definition):
    d = definition.upper().replace(" ", "")
    return d.startswith(("UNIQUE(", "PRIMARYKEY(")) and all(
        c in d for c in ("USER_ID", "REQUEST_ID", "LISTING_ID")
    ) and d.count(",") == 2

assert any(is_save_unique(d) for d in saved_constraints.values()), (
    "BLOCKED: no UNIQUE (user_id, request_id, listing_id) on saved_items. "
    "save_item's duplicate protection depends on it."
)

# 3b — every value the Agent writes must be accepted by the CHECK on action_status.
action_status_def = next(
    (d for d in activity_constraints.values() if "action_status" in d), ""
)
agent_values = [s.value for s in ActionStatus]
missing_action_values = [v for v in agent_values if f"'{v}'" not in action_status_def]

print("Agent writes action_status values:", agent_values)
print("DB CHECK definition:", action_status_def or "(no CHECK on action_status)")
print("Values the DB would reject:", missing_action_values or "none")
assert not missing_action_values, (
    "BLOCKED: agent_activity.action_status CHECK rejects values the Agent writes. "
    "Align ActionStatus in contracts.py with the CHECK above (the DB is the operational truth)."
)
print("PASS Gate 2 — constraints match the Agent")

In [0]:
# Gate 3 — Gold must be consumable by the Agent before any write test.
assert table_exists(GOLD_CANDIDATE_MATCHES), f"BLOCKED: missing {GOLD_CANDIDATE_MATCHES}"
gold = spark.table(GOLD_CANDIDATE_MATCHES)

missing_agent_columns = [c for c in CANDIDATE_MATCH_COLUMNS if c not in gold.columns]
print("Gold columns:", gold.columns)
print("Missing Agent columns:", missing_agent_columns or "none")
assert not missing_agent_columns, (
    "BLOCKED: fix the Gold <-> Agent schema contract before testing save_item. "
    f"Missing: {missing_agent_columns}"
)
print("PASS Gate 3 — Gold matches the Agent contract")

In [0]:
# Stage 12 validation fixture — temporary non-AVAILABLE listing.
# This is test setup only; it is removed after Gate 5.

TEST_UNAVAILABLE_LISTING_ID = "validation-unavailable-listing-001"

if RUN_WRITE_TEST:
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            f"""
            INSERT INTO {LAKEBASE_SCHEMA}.listings (
                listing_id,
                donor_id,
                title,
                description,
                category,
                condition,
                location,
                latitude,
                longitude,
                available_from,
                available_until,
                status,
                created_at,
                updated_at
            )
            VALUES (
                %s,
                'demo-donor-001',
                'Agent Unavailable Validation Fixture',
                'Temporary fixture used only to validate Agent unavailable rejection.',
                'FURNITURE',
                'GOOD',
                'Abu Dhabi',
                24.4976,
                54.4075,
                TIMESTAMP '2026-10-01 00:00:00',
                TIMESTAMP '2026-10-15 23:59:59',
                'WITHDRAWN',
                CURRENT_TIMESTAMP,
                CURRENT_TIMESTAMP
            )
            ON CONFLICT (listing_id) DO UPDATE
            SET
                status = 'WITHDRAWN',
                updated_at = CURRENT_TIMESTAMP
            """,
            (TEST_UNAVAILABLE_LISTING_ID,),
        )

    print(
        "PASS — temporary WITHDRAWN fixture ready:",
        TEST_UNAVAILABLE_LISTING_ID,
    )
else:
    print(
        "REVIEW MODE — unavailable validation fixture not created."
    )

In [0]:
# Gate 4 — choose one deterministic Gold candidate and confirm it exists in Lakebase. Read-only.
from pyspark.sql import functions as F

candidate = (
    gold.orderBy("request_id", F.col("match_score").desc(), "listing_id")
    .select("request_id", "listing_id", "match_score")
    .first()
)
assert candidate is not None, "BLOCKED: Gold has no candidate rows."

REQUEST_ID = str(candidate["request_id"])
LISTING_ID = str(candidate["listing_id"])

request_state = lakebase.get_request(REQUEST_ID)
listing_state = lakebase.get_listing(LISTING_ID)
assert request_state is not None, f"BLOCKED: Gold request_id {REQUEST_ID} is missing from Lakebase."
assert listing_state is not None, f"BLOCKED: Gold listing_id {LISTING_ID} is missing from Lakebase."

USER_ID = request_state.requester_id  # the save must belong to the request's owner

# Any listing that is NOT available, for the rejection test. None found -> that case is skipped.
with connect() as conn, conn.cursor() as cur:
    cur.execute(
        f"SELECT listing_id, status FROM {LAKEBASE_SCHEMA}.listings "
        "WHERE upper(status) <> 'AVAILABLE' ORDER BY listing_id LIMIT 1"
    )
    unavailable_row = cur.fetchone()
UNAVAILABLE_LISTING_ID = unavailable_row[0] if unavailable_row else None

print("Candidate for write test:")
print("  user_id     :", USER_ID)
print("  request_id  :", REQUEST_ID, "| status:", request_state.status)
print("  listing_id  :", LISTING_ID, "| status:", listing_state.status, "| score:", candidate["match_score"])
print("  unavailable :", unavailable_row or "none found (rejection case will be skipped)")
print("PASS Gate 4 — candidate ready. Review it, then set RUN_WRITE_TEST = True in Gate 0a.")

In [0]:
# Gate 5 — Agent WRITE validation
# Writes only when RUN_WRITE_TEST = True.
# Safe to rerun: duplicate protection prevents a second saved_items row.


def saved_count(listing_id):
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            f"""
            SELECT count(*)
            FROM {LAKEBASE_SCHEMA}.saved_items
            WHERE user_id = %s
              AND request_id = %s
              AND listing_id = %s
            """,
            (
                USER_ID,
                REQUEST_ID,
                listing_id,
            ),
        )
        return cur.fetchone()[0]


def recent_audit(listing_id):
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            f"""
            SELECT action_status, created_at
            FROM {LAKEBASE_SCHEMA}.agent_activity
            WHERE user_id = %s
              AND entity_id = %s
              AND tool_name = 'save_item'
            ORDER BY created_at DESC
            LIMIT 10
            """,
            (
                USER_ID,
                listing_id,
            ),
        )
        return cur.fetchall()


# ============================================================
# REVIEW MODE
# ============================================================

if not RUN_WRITE_TEST:

    print(
        "REVIEW MODE — no write executed. "
        "Set RUN_WRITE_TEST = True in Gate 0a "
        "after Gates 0–4 pass."
    )


# ============================================================
# WRITE VALIDATION MODE
# ============================================================

else:

    # --------------------------------------------------------
    # Gate 5a / 5b / 5c
    # Valid save + duplicate protection + audit
    # --------------------------------------------------------

    print(
        "=== GATE 5a–5c — SAVE / DUPLICATE / AUDIT VALIDATION ==="
    )

    before = saved_count(
        LISTING_ID
    )

    first = lakebase.save_item(
        user_id=USER_ID,
        request_id=REQUEST_ID,
        listing_id=LISTING_ID,
    )

    second = lakebase.save_item(
        user_id=USER_ID,
        request_id=REQUEST_ID,
        listing_id=LISTING_ID,
    )

    after = saved_count(
        LISTING_ID
    )

    audit_rows = recent_audit(
        LISTING_ID
    )

    print(
        "Saved rows before / after :",
        before,
        "/",
        after,
    )

    print(
        "First save  :",
        first,
    )

    print(
        "Second save :",
        second,
    )

    print(
        "Audit rows  :",
        audit_rows,
    )


    # --------------------------------------------------------
    # 5a — valid save
    #
    # SUCCESS is expected on a clean first run.
    #
    # REJECTED_DUPLICATE is also valid when this notebook
    # has already been run previously and the saved row exists.
    # --------------------------------------------------------

    assert first.status in (
        ActionStatus.SUCCESS,
        ActionStatus.REJECTED_DUPLICATE,
    ), (
        f"FAILED 5a: unexpected first save result: {first}"
    )


    # --------------------------------------------------------
    # 5b — duplicate must be rejected
    # --------------------------------------------------------

    assert second.status is ActionStatus.REJECTED_DUPLICATE, (
        f"FAILED 5b: expected REJECTED_DUPLICATE, got {second}"
    )


    # There must never be two saved rows for the same
    # user/request/listing business key.
    assert after == 1, (
        "FAILED 5b: expected exactly 1 saved_items row, "
        f"found {after}"
    )


    # --------------------------------------------------------
    # 5c — duplicate attempt must be audited
    # --------------------------------------------------------

    audit_statuses = {
        row[0]
        for row in audit_rows
    }

    assert (
        ActionStatus.REJECTED_DUPLICATE.value
        in audit_statuses
    ), (
        "FAILED 5c: duplicate attempt was not audited "
        "in agent_activity"
    )

    print(
        "PASS 5a–5c — save boundary is idempotent "
        "and duplicate rejection is audited."
    )


    # ========================================================
    # Gate 5d — unavailable listing rejection
    # ========================================================

    print(
        "\n=== GATE 5d — UNAVAILABLE LISTING VALIDATION ==="
    )

    if UNAVAILABLE_LISTING_ID:

        unavailable_before = saved_count(
            UNAVAILABLE_LISTING_ID
        )

        print(
            "Unavailable listing ID :",
            UNAVAILABLE_LISTING_ID,
        )

        print(
            "Saved rows before      :",
            unavailable_before,
        )


        # The fixture must not already have a saved row.
        assert unavailable_before == 0, (
            "FAILED 5d: unavailable validation listing "
            f"already has {unavailable_before} saved row(s)"
        )


        rejected = lakebase.save_item(
            user_id=USER_ID,
            request_id=REQUEST_ID,
            listing_id=UNAVAILABLE_LISTING_ID,
        )


        unavailable_after = saved_count(
            UNAVAILABLE_LISTING_ID
        )

        unavailable_audit = recent_audit(
            UNAVAILABLE_LISTING_ID
        )


        print(
            "Unavailable save       :",
            rejected,
        )

        print(
            "Saved rows after       :",
            unavailable_after,
        )

        print(
            "Unavailable audit      :",
            unavailable_audit,
        )


        # ----------------------------------------------------
        # Agent must reject the listing based on current
        # Lakebase operational truth.
        # ----------------------------------------------------

        assert (
            rejected.status
            is ActionStatus.REJECTED_UNAVAILABLE
        ), (
            "FAILED 5d: expected REJECTED_UNAVAILABLE, "
            f"got {rejected}"
        )


        # ----------------------------------------------------
        # No saved_items row may be created.
        # ----------------------------------------------------

        assert unavailable_after == 0, (
            "FAILED 5d: unavailable listing was saved. "
            f"Found {unavailable_after} saved row(s)."
        )


        # ----------------------------------------------------
        # Rejection must be visible in the audit trail.
        # ----------------------------------------------------

        unavailable_statuses = {
            row[0]
            for row in unavailable_audit
        }

        assert (
            ActionStatus.REJECTED_UNAVAILABLE.value
            in unavailable_statuses
        ), (
            "FAILED 5d: REJECTED_UNAVAILABLE "
            "was not recorded in agent_activity"
        )


        print(
            "PASS 5d — unavailable listing rejected, "
            "no saved_item created, audit confirmed."
        )


    else:

        raise AssertionError(
            "FAILED 5d: no non-AVAILABLE listing exists. "
            "Create the temporary WITHDRAWN validation fixture "
            "before Gate 4."
        )


    # ========================================================
    # FINAL GATE 5 PASS
    # ========================================================

    print(
        "\nPASS Gate 5 — save_item validated: "
        "duplicate protection, unavailable rejection, "
        "idempotency and audit all confirmed."
    )

In [0]:
# Cleanup temporary Stage 12 validation fixture.

if RUN_WRITE_TEST:
    with connect() as conn, conn.cursor() as cur:
        cur.execute(
            f"""
            DELETE FROM {LAKEBASE_SCHEMA}.listings
            WHERE listing_id = %s
            """,
            (TEST_UNAVAILABLE_LISTING_ID,),
        )

    print(
        "PASS — temporary unavailable fixture removed from Lakebase."
    )
else:
    print(
        "REVIEW MODE — no temporary fixture cleanup required."
    )

In [0]:
summary = {
    "stage": "agent_save_item_validation",
    "write_test_enabled": RUN_WRITE_TEST,

    "lakebase": {
        "mode": "autoscaling",
        "endpoint": ENDPOINT_NAME,
        "host": PGHOST,
        "database": PGDATABASE,
        "schema": LAKEBASE_SCHEMA,
        "user": PGUSER,
    },

    "gold_contract_ready": not missing_agent_columns,
    "db_action_status_ready": not missing_action_values,

    "candidate": {
        "user_id": USER_ID,
        "request_id": REQUEST_ID,
        "listing_id": LISTING_ID,
    },

    "unavailable_case": (
        "tested"
        if RUN_WRITE_TEST and UNAVAILABLE_LISTING_ID
        else "not run"
    ),

    "status": (
        "PASS"
        if RUN_WRITE_TEST
        else "READY_FOR_WRITE_TEST"
    ),
}

print(
    json.dumps(
        summary,
        indent=2,
        default=str,
    )
)